# How to setup vs code with databricks
## Step 1 : Install databricks extension on your vs code
## Step 2 : Create a databricks free account and create a workspace in your account.
Your workspace url should look like this : https://dbc-e59b7df7-56be.cloud.databricks.com/editor/folders/home?o=1238258517794919
## Step 3 : Paste this workspace url in your vs code 
![workspace_url](/working_with_databricks_and_pyspark/images/workspace_url.png)
You can activate this above option by clicking on this button as shown below
![activate_workspace_url](/working_with_databricks_and_pyspark/images/activate_workspace_url.png)
## Step 4 : Create a python3.11 env using conda env manager
Use this command to create a new env using conda : ```conda create -n databricks python=3.11```
## Step 5 : activate your databricks env and install databricks-connect
Activate your databricks env
```conda activate databricks```

Install databricks-connect
```pip install databricks-connect```
## Step 6 : Setup your databricks creds in the zsh env conf file
Set your databricks workspace url and your auth token 
```bash
export DATABRICKS_HOST="https://<your-workspace>.cloud.databricks.com"
export DATABRICKS_TOKEN="<paste-your-token>"
export DATABRICKS_CLUSTER_ID="0204-153214-vvzq2sph"
```
Reload your zsh shell after setting up the workspace urls and your auth tokens
```source ~/.zshrc ```

In order to get the url of your databricks workspace you need to copy the url from your browser after opening the workspace in databricks in your browser
As far as auth token is concerned then you need to generate it from user > settings > developer > token then there you need to generate your auth token
You can get you databricks cluster id by following the steps below : 
- Open Databricks workspace
- Go to Compute
- Click your cluster
- Look at the URL in your browser:
    - THE URL IN YOUR BROWSER WILL LOOK SOMETHING LIKE THIS  : ```https://dbc-xyz.cloud.databricks.com#cluster/<THIS_IS_CLUSTER_ID>```
## Step 7 : Install some databricks utilities
```bash
pip install databricks-sdk
pip install databricks-cli
```



# Databricks data engineering :
## Data ingestion : 
```bash
Recommended project structure : 
your_project/
│
├── lakeflow/
│   ├── connections/
│   │   └── postgres_conn.yaml
│   ├── ingest/
│   │   └── orders_ingest.yaml
│   └── pipelines/
│       └── README.md
│
├── src/
│   └── your_app/
│       ├── __init__.py
│       ├── bronze_to_silver/
│       │   ├── __init__.py
│       │   └── process_orders.py
│       ├── utils/
│       │   ├── __init__.py
│       │   └── spark_init.py
│       └── config/
│           └── settings.py
│
├── notebooks/
│   ├── exploration/
│   └── debugging/
│
├── tests/
│   ├── test_process_orders.py
│   └── test_utils.py
│
├── jobs/
│   └── process_orders_job.yaml
│
├── databricks.yml
├── setup.py
├── pyproject.toml
├── README.md
└── .gitignore
```

## Project requirements: 
use-case:
- Data about diabetes patients coming from PostgreSQL
- Ingest using LakeFlow (Bronze)
- Transform with PySpark to Silver
- Clean, validate, standardize medical data
- Produce a Gold table for analytics

This answer will include:
- every folder
- every file
- with real content
- production-style Databricks examples
- but not overwhelming — clean + practical

## COMPLETE PROJECT (EVERY FILE FILLED OUT)
### lakeflow related pipelines
#### lakeflow/ (Ingestion config)
I am going to use AWS RDS postgre server for this 
```bash
connections:
  diabetes_pg:
    type: postgresql
    catalog: main
    schema: bronze_diabetes
    options:
      host: "YOUR_PG_HOST"
      port: "5432"
      database: "healthdb"
    credentials:
      username: "{{secrets/pg/username}}"
      password: "{{secrets/pg/password}}"
```
#### lakeflow/ingest/patient_ingest.yaml
This pipeline ingests the raw table from Postgres into Bronze.
```bash
pipeline:
  name: diabetes_patient_ingest
  schedule: daily
  target: main.bronze_diabetes.patients_raw

  read:
    connection: diabetes_pg
    table: public.diabetes_patients

  write:
    format: delta
    mergeKeys:
      - patient_id
```
#### lakeflow/pipelines/README.md

### src/your_app/ (Your PySpark application code)
#### src/your_app/__init__.py : empty file
#### src/your_app/bronze_to_silver/process_patients.py
Bronze → Silver transformation
```python
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, trim, lower, to_date, when
)

def process():
    spark = SparkSession.builder.getOrCreate()

    # Read Bronze table produced by LakeFlow ingestion
    df = spark.read.table("main.bronze_diabetes.patients_raw")

    # CLEANING & STANDARDIZATION
    df_clean = (
        df
        .withColumn("patient_id", trim(col("patient_id")))
        .withColumn("name", trim(col("name")))
        .withColumn("gender",
            lower(trim(col("gender")))
        )
        .withColumn("diagnosis_date",
            to_date(col("diagnosis_date"), "yyyy-MM-dd")
        )
        .withColumn("blood_sugar",
            col("blood_sugar").cast("double")
        )
        # Standardizing gender
        .withColumn(
            "gender",
            when(col("gender").isin("male", "m"), "male")
            .when(col("gender").isin("female", "f"), "female")
            .otherwise("unknown")
        )
        # Handle null or corrupted blood sugar
        .withColumn(
            "blood_sugar",
            when(col("blood_sugar") < 10, None)  # unrealistic values
            .otherwise(col("blood_sugar"))
        )
    )

    # Write to Silver
    df_clean.write.mode("overwrite").format("delta").saveAsTable(
        "main.silver_diabetes.patients_clean"
    )

    print("✅ Silver table created: main.silver_diabetes.patients_clean")
```
#### src/your_app/bronze_to_silver/diabetes_metrics.py
Analytics (Silver → Gold)
```python
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg

def build_metrics():
    spark = SparkSession.builder.getOrCreate()

    df = spark.read.table("main.silver_diabetes.patients_clean")

    metrics = df.groupBy("gender").agg(
        avg(col("blood_sugar")).alias("avg_blood_sugar")
    )

    metrics.write.mode("overwrite").saveAsTable(
        "main.gold_diabetes.patient_metrics"
    )

    print("✅ Gold table created: main.gold_diabetes.patient_metrics")
```
### Utiltities
#### src/your_app/utils/spark_init.py
```python
from pyspark.sql import SparkSession

def get_spark():
    return SparkSession.builder.getOrCreate()
```
#### src/your_app/config/settings.py
```python
BRONZE_TABLE = "main.bronze_diabetes.patients_raw"
SILVER_TABLE = "main.silver_diabetes.patients_clean"
GOLD_TABLE = "main.gold_diabetes.patient_metrics"
```
### notebooks/
You can keep these empty or add testing notebooks.
### tests/
#### tests/test_process_patients.py
```python
def test_dummy():
    assert 1 == 1
```
### jobs/
#### jobs/process_patients_job.yaml
```bash
jobs:
  diabetes_etl:
    tasks:
      - task_key: bronze_to_silver
        python_wheel_task:
          package_name: your_app
          entry_point: your_app.bronze_to_silver.process_patients:process
        compute:
          serverless: true

      - task_key: build_metrics
        depends_on:
          - task_key: bronze_to_silver
        python_wheel_task:
          package_name: your_app
          entry_point: your_app.bronze_to_silver.diabetes_metrics:build_metrics
        compute:
          serverless: true
```
### Packaging & Deployment
#### setup.py
```python
from setuptools import setup, find_packages

setup(
    name="your_app",
    version="0.1.0",
    packages=find_packages("src"),
    package_dir={"": "src"},
)

```
#### pyproject.toml
```bash
[build-system]
requires = ["setuptools"]
build-backend = "setuptools.build_meta"
```
#### databricks.yml
```bash

bundle:
  name: diabetes_project

artifacts:
  diabetes_app:
    type: python_wheel
    build: "python setup.py bdist_wheel"

targets:
  dev:
    mode: development
    default: true
    workspace:
      host: YOUR_DATABRICKS_URL

resources:
  jobs:
    diabetes_etl:
      <<: *dev
```
#### .gitignore
```bash
__pycache__/
*.pyc
*.pyo
dist/
build/
.DS_Store
```
#### Final Summary (Full E2E Flow)
Here’s the whole pipeline, human-style:
PostgreSQL ──> LakeFlow Connection
                │
                ▼
        LakeFlow Ingest Pipeline
                │
                ▼
       Bronze Table (patients_raw)
                │
                ▼
     PySpark transform (clean data)
                │
                ▼
    Silver Table (patients_clean)
                │
                ▼
     PySpark metrics generation
                │
                ▼
      Gold Table (patient_metrics)
- You now have a full production-ready Databricks + LakeFlow + PySpark project
- With clean folder structure
- Every file filled out
- Bronze → Silver → Gold ETL logic included
- Realistic health/diabetes dataset transformations
